## 2026 EY AI & Data Challenge - Landsat Data Extraction Notebook

This notebook demonstrates Landsat data extraction and the creation of an output file to be used by the benchmark notebook. The baseline data is [Landsat Collection 2 Level 2](https://planetarycomputer.microsoft.com/dataset/landsat-c2-l2) data from the MS Planetary Computer catalog. 

<b>Caution</b> ... This notebook requires significant execution time as there are 9319 data points (unique locations and times) used for data extraction from the Landsat archive. The code takes about 7 hours to run to completion on a typical laptop computer and typical internet connection. Lower execution times are likely possible with optimization of the data extraction process and use of cloud computing services. 

### Load Python Dependencies

In [11]:
import warnings
warnings.filterwarnings("ignore")

# Data manipulation and analysis
import numpy as np
import pandas as pd

# Planetary Computer tools for STAC API access and authentication
import pystac_client
import planetary_computer as pc
from odc.stac import stac_load
from pystac.extensions.eo import EOExtension as eo

from datetime import date
from tqdm import tqdm
import time
import os

<h3>Extracting Landsat Data Using API Calls</h3> <p align="justify"> The API-based method allows us to efficiently access <b>Landsat</b> data for specific coordinates and time periods, ensuring scalability and reproducibility of the process. </p> <p align="justify"> Through the API, we can query individual bands or compute indices like <b>NDMI</b> on-the-fly. This approach reduces storage requirements and simplifies data preprocessing, making it ideal for large-scale environmental and water quality analysis. </p>

<p>The <b>compute_Landsat_values</b> function extracts Landsat surface reflectance values for specific sampling locations using a 100 m focal buffer around each point. For each location:</p>

<ul>
  <li>A bounding box (bbox) is created around the latitude and longitude coordinates.</li>
  <li>The Microsoft Planetary Computer API is queried for Landsat-8 Level-2 surface reflectance imagery within the date range.</li>
  <li>The nearest low-cloud (<10% cloud cover) scene is selected, and the specified bands (<b>green</b>, <b>nir08</b>, <b>swir16</b>, <b>swir22</b>) are loaded.</li>
  <li>Median values of the pixels within the bounding box are computed to reduce the effect of noise or outliers.</li>
</ul>

<p><b>Why the buffer value is 0.00089831:</b></p>

<p>We want a ~100 m buffer around each point. At the equator, 1 degree ≈ 110 km. Therefore, the degree equivalent of 100 m is:</p>

<p style="text-align:center;">
  <em>buffer_deg = 100 m / 110,000 m/deg ≈ 0.00089831</em>
</p>

<p>This slightly adjusted value ensures that the buffer approximately matches the pixel resolution of Landsat imagery, capturing a ~100 m area around each sampling location.</p>


In [12]:
# Setup
tqdm.pandas()

def compute_Landsat_values(row):
    lat = row['Latitude']
    lon = row['Longitude']
    date = pd.to_datetime(row['Sample Date'], dayfirst=True, errors='coerce')

    # Buffer size for ~100m 
    bbox_size = 0.00089831  
    bbox = [
        lon - bbox_size / 2,
        lat - bbox_size / 2,
        lon + bbox_size / 2,
        lat + bbox_size / 2
    ]

    catalog = pystac_client.Client.open(
        "https://planetarycomputer.microsoft.com/api/stac/v1",
        modifier=pc.sign_inplace,
    )

    # Wider search range, we'll filter to nearest date later
    search = catalog.search(
        collections=["landsat-c2-l2"],
        bbox=bbox,
        datetime="2011-01-01/2015-12-31",
        query={"eo:cloud_cover": {"lt": 10}},
    )
    
    items = search.item_collection()

    if not items:
        return pd.Series({
            "nir": np.nan, "green": np.nan, "swir16": np.nan, "swir22": np.nan
        })

    try:
        # Convert sample date to UTC
        sample_date_utc = date.tz_localize("UTC") if date.tzinfo is None else date.tz_convert("UTC")

        # Pick the item closest to the sample date
        items = sorted(
            items,
            key=lambda x: abs(pd.to_datetime(x.properties["datetime"]).tz_convert("UTC") - sample_date_utc)
        )
        selected_item = pc.sign(items[0])

        # Load required bands
        bands_of_interest = ["green", "nir08", "swir16", "swir22"]
        data = stac_load([selected_item], bands=bands_of_interest, bbox=bbox).isel(time=0)

        green = data["green"].astype("float")
        nir = data["nir08"].astype("float")
        swir16 = data["swir16"].astype("float")
        swir22 = data["swir22"].astype("float")
        
        # Compute medians
        median_green = float(green.median(skipna=True).values)
        median_nir = float(nir.median(skipna=True).values)
        median_swir16 = float(swir16.median(skipna=True).values)
        median_swir22 = float(swir22.median(skipna=True).values)

        # Replace 0 with NaN
        median_green = median_green if median_green != 0 else np.nan
        median_nir = median_nir if median_nir != 0 else np.nan
        median_swir16 = median_swir16 if median_swir16 != 0 else np.nan
        median_swir22 = median_swir22 if median_swir22 != 0 else np.nan
        
        return pd.Series({
            "nir": median_nir,
            "green": median_green,
            "swir16": median_swir16,
            "swir22": median_swir22,
        })
    
    except Exception as e:
        return pd.Series({
            "nir": np.nan, "green": np.nan, "swir16": np.nan, "swir22": np.nan
        })


### Extracting features for the training dataset

In [13]:
Water_Quality_df=pd.read_csv('water_quality_training_dataset.csv')
Water_Quality_df.head()

,Latitude,Longitude,Sample Date,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus
0,-28.760833,17.730278,02-01-2011,128.912,555.0,10.0
1,-26.861111,28.884722,03-01-2011,74.720,162.9,163.0
2,-26.450000,28.085833,03-01-2011,89.254,573.0,80.0
3,-27.671111,27.236944,03-01-2011,82.000,203.6,101.0
4,-27.356667,27.286389,03-01-2011,56.100,145.1,151.0


In [4]:
Water_Quality_df.shape

(9319, 6)

In [14]:
Water_Quality_df_200 = Water_Quality_df.loc[0:199]
Water_Quality_df_200.shape

(200, 6)

<h3>Note:</h3>
<p>The Landsat data extraction process for all 9,319 locations typically requires 7+ hours when executed in a single run. During long executions, you may occasionally encounter API limits, timeout errors, or request failures. To avoid these interruptions, we recommend running the extraction in smaller batches.</p>

<p>In this notebook, we provide a sample code snippet demonstrating how to extract data for the first 200 locations. Participants are encouraged to follow the same batching approach to extract data for all 9,319 locations safely and efficiently.</p>

<p>We have already executed the full extraction for all 9,319 locations and saved the output to <b>landsat_features_training.csv</b>, which will be used in the benchmark notebook.
Similarly, participants can extract Landsat features in batches, combine the batch outputs, and save the final merged dataset as <b>landsat_features_training.csv</b> to ensure the benchmark notebook runs smoothly.</p>

In [ ]:
# Extract band values from Landsat for training dataset
train_features_path = "landsat_features_training_200.csv"

print("🚀 Running Landsat feature extraction for training data...")
landsat_train_features = Water_Quality_df_200.progress_apply(compute_Landsat_values, axis=1)
landsat_train_features.to_csv(train_features_path, index=False)

🚀 Running Landsat feature extraction for training data...


  0%|          | 0/200 [00:00<?, ?it/s]

100%|██████████| 200/200 [12:45<00:00,  3.83s/it]


<p><b>NDMI and MNDWI Indices:</b></p>
<p>In this notebook, we compute two commonly used water-related indices from the extracted Landsat bands:</p>
<ul>
  <li><b>NDMI (Normalized Difference Moisture Index):</b> Measures vegetation water content and surface moisture. Computed as <em>(NIR - SWIR16) / (NIR + SWIR16)</em>.</li>
  <li><b>MNDWI (Modified Normalized Difference Water Index):</b> Highlights open water features by enhancing water reflectance and suppressing built-up areas. Computed as <em>(Green - SWIR16) / (Green + SWIR16)</em>.</li>
</ul>

<p>An <b>epsilon value</b> (<em>eps = 1e-10</em>) is added in the denominators to avoid division by zero. These indices are widely used in hydrological and water quality analyses for detecting water presence and vegetation moisture levels.</p>


In [ ]:
# Create indices: NDMI and MNDWI
eps = 1e-10
landsat_train_features['NDMI'] = (landsat_train_features['nir'] - landsat_train_features['swir16']) / (landsat_train_features['nir'] + landsat_train_features['swir16'] + eps)
landsat_train_features['MNDWI'] = (landsat_train_features['green'] - landsat_train_features['swir16']) / (landsat_train_features['green'] + landsat_train_features['swir16'] + eps)

In [ ]:
landsat_train_features['Latitude'] = Water_Quality_df['Latitude']
landsat_train_features['Longitude'] = Water_Quality_df['Longitude']
landsat_train_features['Sample Date'] = Water_Quality_df['Sample Date']
landsat_train_features = landsat_train_features[['Latitude', 'Longitude', 'Sample Date', 'nir', 'green', 'swir16', 'swir22', 'NDMI', 'MNDWI']]

In [ ]:
landsat_train_features.to_csv(train_features_path, index=False)

In [ ]:
# Preview File
landsat_train_features.head()

,Latitude,Longitude,Sample Date,nir,green,swir16,swir22,NDMI,MNDWI
0,-28.760833,17.730278,02-01-2011,11190.0,11426.0,7687.5,7645.0,0.185538,0.195595
1,-26.861111,28.884722,03-01-2011,17658.5,9550.0,13746.5,10574.0,0.124566,-0.180134
2,-26.450000,28.085833,03-01-2011,15210.0,10720.0,17974.0,14201.0,-0.083293,-0.252805
3,-27.671111,27.236944,03-01-2011,14887.0,10943.0,13522.0,11403.0,0.048048,-0.105416
4,-27.356667,27.286389,03-01-2011,16828.5,9502.5,12665.5,9643.0,0.141147,-0.142683


### Extracting features for the validation dataset

In [ ]:
Validation_df=pd.read_csv('submission_template.csv')
Validation_df.head()

,Latitude,Longitude,Sample Date,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus
0,-32.043333,27.822778,01-09-2014,NaN,NaN,NaN
1,-33.329167,26.077500,16-09-2015,NaN,NaN,NaN
2,-32.991639,27.640028,07-05-2015,NaN,NaN,NaN
3,-34.096389,24.439167,07-02-2012,NaN,NaN,NaN
4,-32.000556,28.581667,01-10-2014,NaN,NaN,NaN


In [ ]:
Validation_df.shape

(200, 6)

In [ ]:
# Extract band values from Landsat for submission dataset
val_features_path = "landsat_features_validation.csv"

print("🚀 Running Landsat feature extraction for validation data...")
landsat_val_features = Validation_df.progress_apply(compute_Landsat_values, axis=1)
landsat_val_features.to_csv(val_features_path, index=False)

🚀 Running Landsat feature extraction for validation data...


  0%|          | 0/200 [00:00<?, ?it/s]

100%|██████████| 200/200 [17:51<00:00,  5.36s/it]


In [ ]:
# Create indices: NDMI and MNDWI
eps = 1e-10
landsat_val_features['NDMI'] = (landsat_val_features['nir'] - landsat_val_features['swir16']) / (landsat_val_features['nir'] + landsat_val_features['swir16'])
landsat_val_features['MNDWI'] = (landsat_val_features['green'] - landsat_val_features['swir16']) / (landsat_val_features['green'] + landsat_val_features['swir16'] + eps)

In [ ]:
landsat_val_features['Latitude'] = Validation_df['Latitude']
landsat_val_features['Longitude'] = Validation_df['Longitude']
landsat_val_features['Sample Date'] = Validation_df['Sample Date']
landsat_val_features = landsat_val_features[['Latitude', 'Longitude', 'Sample Date', 'nir', 'green', 'swir16', 'swir22', 'NDMI', 'MNDWI']]

In [ ]:
landsat_val_features.to_csv(val_features_path, index=False)

In [ ]:
# Preview File
landsat_val_features.head()

,Latitude,Longitude,Sample Date,nir,green,swir16,swir22,NDMI,MNDWI
0,-32.043333,27.822778,01-09-2014,15229.0,12868.0,14797.0,12421.0,0.014388,-0.069727
1,-33.329167,26.077500,16-09-2015,NaN,NaN,NaN,NaN,NaN,NaN
2,-32.991639,27.640028,07-05-2015,16221.0,9304.5,12536.5,9958.0,0.128123,-0.147979
3,-34.096389,24.439167,07-02-2012,NaN,NaN,NaN,NaN,NaN,NaN
4,-32.000556,28.581667,01-10-2014,9125.0,11100.5,9455.0,8711.0,-0.017761,0.080052


### 🔴 PRIORITY 2: Multi-Buffer Landsat Extraction 
**Expected API time per cell: ~10-15 minutes for 50 locations**

In [15]:
# PREPARATION: Generate extraction task lists for Landsat_Data_Extraction_Notebook.ipynb
# Run this cell first, then copy the outputs to the extraction notebook

import pandas as pd

# Load location data
locations = pd.read_csv('water_quality_training_dataset.csv')[['Latitude', 'Longitude']].drop_duplicates()
val_locations = pd.read_csv('../data/landsat_features_validation.csv')[['Latitude', 'Longitude']].drop_duplicates()
all_locations = pd.concat([locations, val_locations]).drop_duplicates().reset_index(drop=True)

print(f"Total unique locations to extract: {len(all_locations)}")

# Create batches for API-safe extraction (50 locations per batch = ~10-15min per cell)
BATCH_SIZE = 50
batches = []
for i in range(0, len(all_locations), BATCH_SIZE):
    batch_end = min(i + BATCH_SIZE, len(all_locations))
    batch = all_locations.iloc[i:batch_end].copy()
    batch['batch_id'] = f"batch_{i:03d}_{batch_end:03d}"
    batches.append(batch)
    
print(f"\nCreated {len(batches)} batches of ≤{BATCH_SIZE} locations each")
print("Each batch should complete in 10-15 minutes")

# Save batch metadata
batch_info = pd.DataFrame({
    'batch_id': [f"batch_{i:03d}_{min(i+BATCH_SIZE, len(all_locations)):03d}" for i in range(0, len(all_locations), BATCH_SIZE)],
    'start_idx': range(0, len(all_locations), BATCH_SIZE),
    'end_idx': [min(i+BATCH_SIZE, len(all_locations)) for i in range(0, len(all_locations), BATCH_SIZE)],
    'n_locations': [min(BATCH_SIZE, len(all_locations)-i) for i in range(0, len(all_locations), BATCH_SIZE)]
})

print(f"\nBatch breakdown:")
print(batch_info.to_string(index=False))

# Export for use in extraction notebooks
all_locations.to_csv('extraction_locations_master.csv', index=False)
batch_info.to_csv('extraction_batch_plan.csv', index=False)

print(f"\n✅ Extraction plan saved:")
print(f"   📁 extraction_locations_master.csv ({len(all_locations)} locations)")
print(f"   📁 extraction_batch_plan.csv ({len(batches)} batches)")

Total unique locations to extract: 186

Created 4 batches of ≤50 locations each
Each batch should complete in 10-15 minutes

Batch breakdown:
     batch_id  start_idx  end_idx  n_locations
batch_000_050          0       50           50
batch_050_100         50      100           50
batch_100_150        100      150           50
batch_150_186        150      186           36

✅ Extraction plan saved:
   📁 extraction_locations_master.csv (186 locations)
   📁 extraction_batch_plan.csv (4 batches)


#### Multi-Buffer Extraction Functions


In [16]:
# MULTI-BUFFER LANDSAT EXTRACTION FUNCTIONS

def compute_Landsat_values_custom_buffer(row, buffer_meters=100):
    """Modified version of compute_Landsat_values with custom buffer size"""
    lat = row['Latitude']
    lon = row['Longitude']
    date = pd.to_datetime(row['Sample Date'], dayfirst=True, errors='coerce')

    # Convert buffer from meters to degrees
    # At equator: 1 degree ≈ 110 km, so buffer_deg = buffer_meters / 110,000
    bbox_size = buffer_meters / 110000  
    bbox = [
        lon - bbox_size / 2,
        lat - bbox_size / 2,
        lon + bbox_size / 2,
        lat + bbox_size / 2
    ]

    catalog = pystac_client.Client.open(
        "https://planetarycomputer.microsoft.com/api/stac/v1",
        modifier=pc.sign_inplace,
    )

    search = catalog.search(
        collections=["landsat-c2-l2"],
        bbox=bbox,
        datetime="2011-01-01/2015-12-31",
        query={"eo:cloud_cover": {"lt": 10}},
    )
    
    items = search.item_collection()

    if not items:
        return pd.Series({
            "nir": np.nan, "green": np.nan, "swir16": np.nan, "swir22": np.nan
        })

    try:
        # Convert sample date to UTC
        sample_date_utc = date.tz_localize("UTC") if date.tzinfo is None else date.tz_convert("UTC")

        # Pick the item closest to the sample date
        items = sorted(
            items,
            key=lambda x: abs(pd.to_datetime(x.properties["datetime"]).tz_convert("UTC") - sample_date_utc)
        )
        selected_item = pc.sign(items[0])

        # Load required bands
        bands_of_interest = ["green", "nir08", "swir16", "swir22"]
        data = stac_load([selected_item], bands=bands_of_interest, bbox=bbox).isel(time=0)

        green = data["green"].astype("float")
        nir = data["nir08"].astype("float")
        swir16 = data["swir16"].astype("float")
        swir22 = data["swir22"].astype("float")
        
        # Compute medians
        median_green = float(green.median(skipna=True).values)
        median_nir = float(nir.median(skipna=True).values)
        median_swir16 = float(swir16.median(skipna=True).values)
        median_swir22 = float(swir22.median(skipna=True).values)

        # Replace 0 with NaN
        median_green = median_green if median_green != 0 else np.nan
        median_nir = median_nir if median_nir != 0 else np.nan
        median_swir16 = median_swir16 if median_swir16 != 0 else np.nan
        median_swir22 = median_swir22 if median_swir22 != 0 else np.nan
        
        return pd.Series({
            "nir": median_nir,
            "green": median_green,
            "swir16": median_swir16,
            "swir22": median_swir22,
        })
    
    except Exception as e:
        print(f"Error processing {lat}, {lon}: {e}")
        return pd.Series({
            "nir": np.nan, "green": np.nan, "swir16": np.nan, "swir22": np.nan
        })

def extract_landsat_features_batch(locations_df, buffer_meters=100, add_indices=True):
    """Extract Landsat features for a batch of locations with custom buffer"""
    print(f"🚀 Extracting Landsat features for {len(locations_df)} locations (buffer: {buffer_meters}m)...")
    
    # Apply extraction function with custom buffer
    landsat_features = locations_df.progress_apply(
        lambda row: compute_Landsat_values_custom_buffer(row, buffer_meters), 
        axis=1
    )
    
    # Add indices if requested
    if add_indices:
        eps = 1e-10
        landsat_features['NDMI'] = (
            (landsat_features['nir'] - landsat_features['swir16']) / 
            (landsat_features['nir'] + landsat_features['swir16'] + eps)
        )
        landsat_features['MNDWI'] = (
            (landsat_features['green'] - landsat_features['swir16']) / 
            (landsat_features['green'] + landsat_features['swir16'] + eps)
        )
    
    # Add coordinate and date columns
    landsat_features['Latitude'] = locations_df['Latitude']
    landsat_features['Longitude'] = locations_df['Longitude']
    landsat_features['Sample Date'] = locations_df['Sample Date']
    
    # Reorder columns
    column_order = ['Latitude', 'Longitude', 'Sample Date', 'nir', 'green', 'swir16', 'swir22']
    if add_indices:
        column_order.extend(['NDMI', 'MNDWI'])
    
    landsat_features = landsat_features[column_order]
    
    print(f"✅ Extraction complete: {len(landsat_features)} locations processed")
    return landsat_features

print("✅ Multi-buffer extraction functions loaded")


✅ Multi-buffer extraction functions loaded


#### 🔴 50m Buffer Extraction (API-Safe Batches)
**Process locations in 50-location batches to avoid API timeouts**


In [17]:
# Load batch plan and locations
try:
    batch_info = pd.read_csv('extraction_batch_plan.csv')
    all_locations = pd.read_csv('extraction_locations_master.csv')
    print(f"📋 Loaded batch plan: {len(batch_info)} batches, {len(all_locations)} total locations")
except FileNotFoundError:
    print("⚠️ Run the preparation cell in Modeling_Optimization.ipynb first!")
    batch_info = None

if batch_info is not None:
    print(f"\nFirst few batches:")
    print(batch_info.head().to_string(index=False))


📋 Loaded batch plan: 4 batches, 186 total locations

First few batches:
     batch_id  start_idx  end_idx  n_locations
batch_000_050          0       50           50
batch_050_100         50      100           50
batch_100_150        100      150           50
batch_150_186        150      186           36


In [15]:
# EXTRACT 50m BUFFER - BATCH 1 (Locations 0-49)
# Expected time: ~10-15 minutes

if 'all_locations' in globals():
    BUFFER_SIZE = 50
    BATCH_START = 0
    BATCH_END = 50
    
    batch_locations = all_locations.iloc[BATCH_START:BATCH_END].copy()
    batch_locations['Sample Date'] = '2013-06-15'  # Use middle date for extraction
    
    print(f"Processing batch {BATCH_START}-{BATCH_END} with {BUFFER_SIZE}m buffer...")
    start_time = time.time()
    
    features_50m_batch1 = extract_landsat_features_batch(batch_locations, buffer_meters=BUFFER_SIZE)
    
    # Save immediately
    output_file = f'landsat_{BUFFER_SIZE}m_batch_{BATCH_START:03d}_{BATCH_END:03d}.csv'
    features_50m_batch1.to_csv(output_file, index=False)
    
    elapsed = time.time() - start_time
    print(f"✅ {output_file} saved ({elapsed:.1f}s, {len(batch_locations)} locations)")
else:
    print("⚠️ Run the batch preparation cell first!")


Processing batch 0-50 with 50m buffer...
🚀 Extracting Landsat features for 50 locations (buffer: 50m)...


100%|██████████| 50/50 [03:55<00:00,  4.71s/it]

✅ Extraction complete: 50 locations processed
✅ landsat_50m_batch_000_050.csv saved (235.8s, 50 locations)


In [16]:
# EXTRACT 50m BUFFER - BATCH 2 (Locations 50-99) 
# Expected time: ~10-15 minutes

if 'all_locations' in globals():
    BUFFER_SIZE = 50
    BATCH_START = 50
    BATCH_END = 100
    
    batch_locations = all_locations.iloc[BATCH_START:BATCH_END].copy()
    batch_locations['Sample Date'] = '2013-06-15'
    
    print(f"Processing batch {BATCH_START}-{BATCH_END} with {BUFFER_SIZE}m buffer...")
    start_time = time.time()
    
    features_50m_batch2 = extract_landsat_features_batch(batch_locations, buffer_meters=BUFFER_SIZE)
    
    output_file = f'landsat_{BUFFER_SIZE}m_batch_{BATCH_START:03d}_{BATCH_END:03d}.csv'
    features_50m_batch2.to_csv(output_file, index=False)
    
    elapsed = time.time() - start_time
    print(f"✅ {output_file} saved ({elapsed:.1f}s, {len(batch_locations)} locations)")
else:
    print("⚠️ Run the batch preparation cell first!")


Processing batch 50-100 with 50m buffer...
🚀 Extracting Landsat features for 50 locations (buffer: 50m)...


100%|██████████| 50/50 [05:08<00:00,  6.18s/it]

✅ Extraction complete: 50 locations processed
✅ landsat_50m_batch_050_100.csv saved (309.1s, 50 locations)


In [17]:
# EXTRACT 50m BUFFER - BATCH 3 (Locations 100-149)
# Expected time: ~5 minutes (based on your results)

if 'all_locations' in globals():
    BUFFER_SIZE = 50
    BATCH_START = 100
    BATCH_END = 150
    
    if BATCH_END <= len(all_locations):  # Check if batch exists
        batch_locations = all_locations.iloc[BATCH_START:BATCH_END].copy()
        batch_locations['Sample Date'] = '2013-06-15'
        
        print(f"Processing batch {BATCH_START}-{BATCH_END} with {BUFFER_SIZE}m buffer...")
        start_time = time.time()
        
        features_50m_batch3 = extract_landsat_features_batch(batch_locations, buffer_meters=BUFFER_SIZE)
        
        output_file = f'landsat_{BUFFER_SIZE}m_batch_{BATCH_START:03d}_{BATCH_END:03d}.csv'
        features_50m_batch3.to_csv(output_file, index=False)
        
        elapsed = time.time() - start_time
        print(f"✅ {output_file} saved ({elapsed:.1f}s, {len(batch_locations)} locations)")
    else:
        print(f"⭐ Batch {BATCH_START}-{BATCH_END} exceeds dataset size ({len(all_locations)} locations)")
else:
    print("⚠️ Run the batch preparation cell first!")

Processing batch 100-150 with 50m buffer...
🚀 Extracting Landsat features for 50 locations (buffer: 50m)...


100%|██████████| 50/50 [04:15<00:00,  5.12s/it]

✅ Extraction complete: 50 locations processed
✅ landsat_50m_batch_100_150.csv saved (256.0s, 50 locations)


In [19]:
# EXTRACT 50m BUFFER - BATCH 4 (Locations 150-199)
if 'all_locations' in globals():
    BUFFER_SIZE = 50
    BATCH_START = 150
    BATCH_END = 186
    
    if BATCH_END <= len(all_locations):
        batch_locations = all_locations.iloc[BATCH_START:BATCH_END].copy()
        batch_locations['Sample Date'] = '2013-06-15'
        
        print(f"Processing batch {BATCH_START}-{BATCH_END} with {BUFFER_SIZE}m buffer...")
        start_time = time.time()
        
        features_50m_batch4 = extract_landsat_features_batch(batch_locations, buffer_meters=BUFFER_SIZE)
        
        output_file = f'landsat_{BUFFER_SIZE}m_batch_{BATCH_START:03d}_{BATCH_END:03d}.csv'
        features_50m_batch4.to_csv(output_file, index=False)
        
        elapsed = time.time() - start_time
        print(f"✅ {output_file} saved ({elapsed:.1f}s, {len(batch_locations)} locations)")
    else:
        print(f"⭐ Batch {BATCH_START}-{BATCH_END} exceeds dataset size ({len(all_locations)} locations)")
else:
    print("⚠️ Run the batch preparation cell first!")

Processing batch 150-186 with 50m buffer...
🚀 Extracting Landsat features for 36 locations (buffer: 50m)...


100%|██████████| 36/36 [02:25<00:00,  4.04s/it]

✅ Extraction complete: 36 locations processed
✅ landsat_50m_batch_150_186.csv saved (145.4s, 36 locations)


#### 🔴 150m Buffer Extraction (Larger Context)

In [20]:
# EXTRACT 150m BUFFER - BATCH 1 (Locations 0-49)
# Expected time: ~10-15 minutes

if 'all_locations' in globals():
    BUFFER_SIZE = 150
    BATCH_START = 0
    BATCH_END = 50
    
    batch_locations = all_locations.iloc[BATCH_START:BATCH_END].copy()
    batch_locations['Sample Date'] = '2013-06-15'
    
    print(f"Processing batch {BATCH_START}-{BATCH_END} with {BUFFER_SIZE}m buffer...")
    start_time = time.time()
    
    features_150m_batch1 = extract_landsat_features_batch(batch_locations, buffer_meters=BUFFER_SIZE)
    
    output_file = f'landsat_{BUFFER_SIZE}m_batch_{BATCH_START:03d}_{BATCH_END:03d}.csv'
    features_150m_batch1.to_csv(output_file, index=False)
    
    elapsed = time.time() - start_time
    print(f"✅ {output_file} saved ({elapsed:.1f}s, {len(batch_locations)} locations)")
else:
    print("⚠️ Run the batch preparation cell first!")

Processing batch 0-50 with 150m buffer...
🚀 Extracting Landsat features for 50 locations (buffer: 150m)...


100%|██████████| 50/50 [03:36<00:00,  4.32s/it]

✅ Extraction complete: 50 locations processed
✅ landsat_150m_batch_000_050.csv saved (216.2s, 50 locations)


In [18]:
# EXTRACT 150m BUFFER - BATCH 2 (Locations 50-99)
if 'all_locations' in globals():
    BUFFER_SIZE = 150
    BATCH_START = 50
    BATCH_END = 100
    
    batch_locations = all_locations.iloc[BATCH_START:BATCH_END].copy()
    batch_locations['Sample Date'] = '2013-06-15'
    
    print(f"Processing batch {BATCH_START}-{BATCH_END} with {BUFFER_SIZE}m buffer...")
    start_time = time.time()
    
    features_150m_batch2 = extract_landsat_features_batch(batch_locations, buffer_meters=BUFFER_SIZE)
    
    output_file = f'landsat_{BUFFER_SIZE}m_batch_{BATCH_START:03d}_{BATCH_END:03d}.csv'
    features_150m_batch2.to_csv(output_file, index=False)
    
    elapsed = time.time() - start_time
    print(f"✅ {output_file} saved ({elapsed:.1f}s, {len(batch_locations)} locations)")
else:
    print("⚠️ Run the batch preparation cell first!")

Processing batch 50-100 with 150m buffer...
🚀 Extracting Landsat features for 50 locations (buffer: 150m)...


  0%|          | 0/50 [00:00<?, ?it/s]

100%|██████████| 50/50 [05:34<00:00,  6.68s/it]

✅ Extraction complete: 50 locations processed
✅ landsat_150m_batch_050_100.csv saved (334.2s, 50 locations)


In [21]:
# EXTRACT 150m BUFFER - BATCH 3 (Locations 100-149)
if 'all_locations' in globals():
    BUFFER_SIZE = 150
    BATCH_START = 100
    BATCH_END = 150
    
    batch_locations = all_locations.iloc[BATCH_START:BATCH_END].copy()
    batch_locations['Sample Date'] = '2013-06-15'
    
    print(f"Processing batch {BATCH_START}-{BATCH_END} with {BUFFER_SIZE}m buffer...")
    start_time = time.time()
    
    features_150m_batch3 = extract_landsat_features_batch(batch_locations, buffer_meters=BUFFER_SIZE)
    
    output_file = f'landsat_{BUFFER_SIZE}m_batch_{BATCH_START:03d}_{BATCH_END:03d}.csv'
    features_150m_batch3.to_csv(output_file, index=False)
    
    elapsed = time.time() - start_time
    print(f"✅ {output_file} saved ({elapsed:.1f}s, {len(batch_locations)} locations)")
else:
    print("⚠️ Run the batch preparation cell first!")

Processing batch 100-150 with 150m buffer...
🚀 Extracting Landsat features for 50 locations (buffer: 150m)...


100%|██████████| 50/50 [03:49<00:00,  4.59s/it]

✅ Extraction complete: 50 locations processed
✅ landsat_150m_batch_100_150.csv saved (229.8s, 50 locations)


In [22]:
# EXTRACT 150m BUFFER - BATCH 4 (Locations 150-199)
if 'all_locations' in globals():
    BUFFER_SIZE = 150
    BATCH_START = 150
    BATCH_END = 200
    
    batch_locations = all_locations.iloc[BATCH_START:BATCH_END].copy()
    batch_locations['Sample Date'] = '2013-06-15'
    
    print(f"Processing batch {BATCH_START}-{BATCH_END} with {BUFFER_SIZE}m buffer...")
    start_time = time.time()
    
    features_150m_batch4 = extract_landsat_features_batch(batch_locations, buffer_meters=BUFFER_SIZE)
    
    output_file = f'landsat_{BUFFER_SIZE}m_batch_{BATCH_START:03d}_{BATCH_END:03d}.csv'
    features_150m_batch4.to_csv(output_file, index=False)
    
    elapsed = time.time() - start_time
    print(f"✅ {output_file} saved ({elapsed:.1f}s, {len(batch_locations)} locations)")
else:
    print("⚠️ Run the batch preparation cell first!")

Processing batch 150-200 with 150m buffer...
🚀 Extracting Landsat features for 36 locations (buffer: 150m)...


100%|██████████| 36/36 [02:56<00:00,  4.91s/it]

✅ Extraction complete: 36 locations processed
✅ landsat_150m_batch_150_200.csv saved (176.7s, 36 locations)


#### 🔴 200m Buffer Extraction (Wide Context)
**Larger buffer captures more landscape context and upstream influences**

In [23]:
# EXTRACT 200m BUFFER - ALL BATCHES (Complete Extraction)
# Process all locations in one efficient loop for 200m buffer

if 'all_locations' in globals():
    BUFFER_SIZE = 200
    
    print(f"🚀 Starting 200m buffer extraction for all {len(all_locations)} locations...")
    print("Processing in batches of 50...")
    
    # Process all locations in batches of 50
    for batch_num, start_idx in enumerate(range(0, len(all_locations), 50), 1):
        end_idx = min(start_idx + 50, len(all_locations))
        
        batch_locations = all_locations.iloc[start_idx:end_idx].copy()
        batch_locations['Sample Date'] = '2013-06-15'
        
        print(f"\n📊 Batch {batch_num}: Processing locations {start_idx}-{end_idx} ({len(batch_locations)} locations)")
        start_time = time.time()
        
        features_200m = extract_landsat_features_batch(batch_locations, buffer_meters=BUFFER_SIZE)
        
        output_file = f'landsat_{BUFFER_SIZE}m_batch_{start_idx:03d}_{end_idx:03d}.csv'
        features_200m.to_csv(output_file, index=False)
        
        elapsed = time.time() - start_time
        print(f"   ✅ {output_file} saved ({elapsed:.1f}s)")
    
    print(f"\n🎉 200m buffer extraction COMPLETE!")
    print(f"Total batches created: {len(range(0, len(all_locations), 50))}")
else:
    print("⚠️ Run the batch preparation cell first!")

🚀 Starting 200m buffer extraction for all 186 locations...
Processing in batches of 50...

📊 Batch 1: Processing locations 0-50 (50 locations)
🚀 Extracting Landsat features for 50 locations (buffer: 200m)...


100%|██████████| 50/50 [04:20<00:00,  5.21s/it]


✅ Extraction complete: 50 locations processed
   ✅ landsat_200m_batch_000_050.csv saved (260.4s)

📊 Batch 2: Processing locations 50-100 (50 locations)
🚀 Extracting Landsat features for 50 locations (buffer: 200m)...


100%|██████████| 50/50 [03:59<00:00,  4.79s/it]


✅ Extraction complete: 50 locations processed
   ✅ landsat_200m_batch_050_100.csv saved (239.8s)

📊 Batch 3: Processing locations 100-150 (50 locations)
🚀 Extracting Landsat features for 50 locations (buffer: 200m)...


100%|██████████| 50/50 [04:02<00:00,  4.84s/it]


✅ Extraction complete: 50 locations processed
   ✅ landsat_200m_batch_100_150.csv saved (242.2s)

📊 Batch 4: Processing locations 150-186 (36 locations)
🚀 Extracting Landsat features for 36 locations (buffer: 200m)...


100%|██████████| 36/36 [02:45<00:00,  4.59s/it]

✅ Extraction complete: 36 locations processed
   ✅ landsat_200m_batch_150_186.csv saved (165.2s)

🎉 200m buffer extraction COMPLETE!
Total batches created: 4


#### 📋 Final Assembly - Combine All Batches
**Run after all extractions complete to create final feature files**

In [24]:
# COMBINE ALL BATCHES - Create Final Feature Files
# Run this after all extractions complete

import glob
from pathlib import Path

def combine_landsat_batches():
    """Combine all batch CSV files into final feature datasets"""
    
    print("🔄 Combining all Landsat extraction batches...\n")
    
    for buffer in [50, 150, 200]:
        pattern = f'landsat_{buffer}m_batch_*.csv'
        batch_files = sorted(glob.glob(pattern))
        
        print(f"📊 {buffer}m buffer: Found {len(batch_files)} batch files")
        
        if not batch_files:
            print(f"   ⚠️ No batch files found for {buffer}m buffer")
            continue
        
        # Load and combine all batches
        all_batches = []
        total_locations = 0
        
        for file in batch_files:
            try:
                batch = pd.read_csv(file)
                all_batches.append(batch)
                total_locations += len(batch)
                print(f"   ✅ {Path(file).name}: {len(batch)} locations")
            except Exception as e:
                print(f"   ❌ Error loading {Path(file).name}: {e}")
        
        if all_batches:
            # Combine all batches
            combined = pd.concat(all_batches, ignore_index=True)
            
            # Remove duplicates (if any)
            before = len(combined)
            combined = combined.drop_duplicates(subset=['Latitude', 'Longitude', 'Sample Date']).reset_index(drop=True)
            after = len(combined)
            
            if before > after:
                print(f"   🔧 Removed {before-after} duplicate rows")
            
            # Save combined dataset
            output_file = f'landsat_features_{buffer}m_combined.csv'
            combined.to_csv(output_file, index=False)
            
            print(f"   🎉 {output_file} saved: {len(combined)} locations × {len(combined.columns)} features")
            
            # Display sample
            print(f"   📋 Sample data:")
            print(combined.head(2).to_string(index=False))
        
        print()
    
    print("✅ Multi-buffer Landsat extraction COMPLETE!")
    print("\n📁 Final files created:")
    for buffer in [50, 150, 200]:
        filename = f'landsat_features_{buffer}m_combined.csv'
        if Path(filename).exists():
            size = Path(filename).stat().st_size / 1024 / 1024  # MB
            print(f"   • {filename} ({size:.1f} MB)")
    
    print(f"\n🚀 Next: Use these files in Modeling_Optimization.ipynb for feature comparison")

# Run the combination
combine_landsat_batches()

🔄 Combining all Landsat extraction batches...

📊 50m buffer: Found 4 batch files
   ✅ landsat_50m_batch_000_050.csv: 50 locations
   ✅ landsat_50m_batch_050_100.csv: 50 locations
   ✅ landsat_50m_batch_100_150.csv: 50 locations
   ✅ landsat_50m_batch_150_186.csv: 36 locations
   🎉 landsat_features_50m_combined.csv saved: 186 locations × 9 features
   📋 Sample data:
  Latitude  Longitude Sample Date     nir   green  swir16  swir22     NDMI     MNDWI
-28.760833  17.730278  2013-06-15     NaN     NaN     NaN     NaN      NaN       NaN
-26.861111  28.884722  2013-06-15 15417.0 10408.5 18224.0 14732.5 -0.08344 -0.272959

📊 150m buffer: Found 4 batch files
   ✅ landsat_150m_batch_000_050.csv: 50 locations
   ✅ landsat_150m_batch_050_100.csv: 50 locations
   ✅ landsat_150m_batch_100_150.csv: 50 locations
   ✅ landsat_150m_batch_150_200.csv: 36 locations
   🎉 landsat_features_150m_combined.csv saved: 186 locations × 9 features
   📋 Sample data:
  Latitude  Longitude Sample Date     nir   green

In [25]:
# 📊 PROGRESS TRACKER - Check Completed Extractions
# Run this cell to see which batches are already completed

import glob
from pathlib import Path

def check_extraction_progress():
    """Check which batch files have been created"""
    print("🔍 Checking extraction progress...\n")
    
    for buffer in [50, 100, 150, 200]:
        pattern = f'landsat_{buffer}m_batch_*.csv'
        files = sorted(glob.glob(pattern))
        
        print(f"📊 {buffer}m buffer: {len(files)} batch files found")
        
        if files:
            total_locations = 0
            for file in files[:5]:  # Show first 5 files
                try:
                    df = pd.read_csv(file)
                    total_locations += len(df)
                    print(f"   ✅ {Path(file).name}: {len(df)} locations")
                except:
                    print(f"   ❌ {Path(file).name}: Error reading file")
            
            if len(files) > 5:
                print(f"   ... and {len(files) - 5} more files")
                
            print(f"   📈 Total locations extracted: {total_locations}")
        else:
            print(f"   ⚪ No {buffer}m buffer extractions found")
        
        print()
    
    # Check if all_locations exists for comparison
    try:
        all_locs = pd.read_csv('extraction_locations_master.csv')
        print(f"🎯 Target: {len(all_locs)} total unique locations to extract")
    except:
        print("ℹ️ Run Modeling_Optimization.ipynb preparation cell to generate target counts")

check_extraction_progress()

🔍 Checking extraction progress...

📊 50m buffer: 4 batch files found
   ✅ landsat_50m_batch_000_050.csv: 50 locations
   ✅ landsat_50m_batch_050_100.csv: 50 locations
   ✅ landsat_50m_batch_100_150.csv: 50 locations
   ✅ landsat_50m_batch_150_186.csv: 36 locations
   📈 Total locations extracted: 186

📊 100m buffer: 0 batch files found
   ⚪ No 100m buffer extractions found

📊 150m buffer: 4 batch files found
   ✅ landsat_150m_batch_000_050.csv: 50 locations
   ✅ landsat_150m_batch_050_100.csv: 50 locations
   ✅ landsat_150m_batch_100_150.csv: 50 locations
   ✅ landsat_150m_batch_150_200.csv: 36 locations
   📈 Total locations extracted: 186

📊 200m buffer: 4 batch files found
   ✅ landsat_200m_batch_000_050.csv: 50 locations
   ✅ landsat_200m_batch_050_100.csv: 50 locations
   ✅ landsat_200m_batch_100_150.csv: 50 locations
   ✅ landsat_200m_batch_150_186.csv: 36 locations
   📈 Total locations extracted: 186

🎯 Target: 186 total unique locations to extract


---

## 🚀 TRAINING DATASET EXTRACTION (9,319 Data Points)

**What you completed above**: Validation dataset (186 unique locations)  
**What we're doing now**: Training dataset (9,319 data points with dates)

The training dataset requires significantly more extraction time due to the larger size. We'll use the same batching strategy (50 locations per batch) to avoid API timeouts.

**Estimated time**: ~6-8 hours total (spread across multiple sessions)

In [26]:
# PREPARATION: Training Dataset Batch Setup
# Load training dataset and create extraction batches

# Load the full training dataset (9,319 rows)
training_df = pd.read_csv('water_quality_training_dataset.csv')
print(f"📋 Training dataset loaded: {len(training_df)} data points")
print(f"Columns: {list(training_df.columns)}")
print(f"Date range: {training_df['Sample Date'].min()} to {training_df['Sample Date'].max()}")

# Create batches for training data extraction
TRAINING_BATCH_SIZE = 1000  # 1000 locations per batch for efficient processing
training_batches = []

for i in range(0, len(training_df), TRAINING_BATCH_SIZE):
    batch_end = min(i + TRAINING_BATCH_SIZE, len(training_df))
    batch = training_df.iloc[i:batch_end].copy()
    batch_id = f"train_batch_{i:04d}_{batch_end:04d}"
    training_batches.append({
        'batch_id': batch_id,
        'start_idx': i,
        'end_idx': batch_end,
        'n_locations': len(batch),
        'data': batch
    })

print(f"\n🔄 Created {len(training_batches)} training batches")
print(f"Each batch: ≤{TRAINING_BATCH_SIZE} locations (~3-4 hours per buffer)")
print(f"Total estimated time per buffer: ~{len(training_batches) * 3.5:.1f} hours")

# Show batch breakdown
print(f"\nFirst 5 batches:")
for i in range(min(5, len(training_batches))):
    batch = training_batches[i]
    print(f"  {batch['batch_id']}: {batch['n_locations']} locations ({batch['start_idx']}-{batch['end_idx']})")

if len(training_batches) > 5:
    print(f"  ... and {len(training_batches) - 5} more batches")

print(f"\n✅ Training extraction setup complete")

📋 Training dataset loaded: 9319 data points
Columns: ['Latitude', 'Longitude', 'Sample Date', 'Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']
Date range: 01-01-2013 to 31-12-2015

🔄 Created 10 training batches
Each batch: ≤1000 locations (~3-4 hours per buffer)
Total estimated time per buffer: ~35.0 hours

First 5 batches:
  train_batch_0000_1000: 1000 locations (0-1000)
  train_batch_1000_2000: 1000 locations (1000-2000)
  train_batch_2000_3000: 1000 locations (2000-3000)
  train_batch_3000_4000: 1000 locations (3000-4000)
  train_batch_4000_5000: 1000 locations (4000-5000)
  ... and 5 more batches

✅ Training extraction setup complete


### 🔴 Training Dataset: 50m Buffer Extraction
**Process all 9,319 training data points with 50m buffer**

In [28]:
# TRAINING 50m BUFFER - BATCH 1 (Locations 0-999)
# Run this first to test the training extraction workflow

if 'training_batches' in globals() and len(training_batches) > 0:
    BUFFER_SIZE = 50
    batch_idx = 0  # First batch
    
    batch = training_batches[batch_idx]
    batch_data = batch['data']
    
    print(f"🚀 Training Batch {batch_idx + 1}: {batch['batch_id']} ({BUFFER_SIZE}m buffer)")
    print(f"   Processing {len(batch_data)} locations...")
    
    start_time = time.time()
    
    features_train_50m = extract_landsat_features_batch(batch_data, buffer_meters=BUFFER_SIZE)
    
    # Save with training prefix
    output_file = f"landsat_train_{BUFFER_SIZE}m_{batch['batch_id']}.csv"
    features_train_50m.to_csv(output_file, index=False)
    
    elapsed = time.time() - start_time
    print(f"✅ {output_file} saved ({elapsed:.1f}s, {len(batch_data)} locations)")
    
    # Show sample data
    print(f"\n📋 Sample extracted data:")
    print(features_train_50m.head(2).to_string(index=False))
else:
    print("⚠️ Run the training batch preparation cell first!")

🚀 Training Batch 1: train_batch_0000_1000 (50m buffer)
   Processing 1000 locations...
🚀 Extracting Landsat features for 1000 locations (buffer: 50m)...


  0%|          | 1/1000 [00:00<01:19, 12.54it/s]


APIError: HTTPSConnectionPool(host='planetarycomputer.microsoft.com', port=443): Max retries exceeded with url: /api/stac/v1 (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x14f0a6c10>: Failed to resolve 'planetarycomputer.microsoft.com' ([Errno 8] nodename nor servname provided, or not known)"))

In [ ]:
# TRAINING 50m BUFFER - BATCH 2 (Locations 1000-1999)
# Run this first to test the training extraction workflow

if 'training_batches' in globals() and len(training_batches) > 0:
    BUFFER_SIZE = 50
    batch_idx = 1  # Second batch
    
    batch = training_batches[batch_idx]
    batch_data = batch['data']
    
    print(f"🚀 Training Batch {batch_idx + 1}: {batch['batch_id']} ({BUFFER_SIZE}m buffer)")
    print(f"   Processing {len(batch_data)} locations...")
    
    start_time = time.time()
    
    features_train_50m = extract_landsat_features_batch(batch_data, buffer_meters=BUFFER_SIZE)
    
    # Save with training prefix
    output_file = f"landsat_train_{BUFFER_SIZE}m_{batch['batch_id']}.csv"
    features_train_50m.to_csv(output_file, index=False)
    
    elapsed = time.time() - start_time
    print(f"✅ {output_file} saved ({elapsed:.1f}s, {len(batch_data)} locations)")
    
    # Show sample data
    print(f"\n📋 Sample extracted data:")
    print(features_train_50m.head(2).to_string(index=False))
else:
    print("⚠️ Run the training batch preparation cell first!")

In [ ]:
# TRAINING 50m BUFFER - ALL REMAINING BATCHES (Batches 2+)
# Run this to process all remaining training batches for 50m buffer
# WARNING: This will run for several hours - monitor progress!

if 'training_batches' in globals():
    BUFFER_SIZE = 50
    
    print(f"🚀 Starting 50m buffer extraction for ALL training batches...")
    print(f"Total batches: {len(training_batches)} (≈{len(training_batches) * 3.5:.1f} hours)")
    
    successful_batches = 0
    failed_batches = []
    
    for batch_idx in range(1, len(training_batches)):  # Start from batch 2 (index 1)
        batch = training_batches[batch_idx]
        batch_data = batch['data']
        
        output_file = f"landsat_train_{BUFFER_SIZE}m_{batch['batch_id']}.csv"
        
        # Skip if already exists
        if os.path.exists(output_file):
            print(f"⏭️  Batch {batch_idx + 1}: {output_file} already exists, skipping...")
            successful_batches += 1
            continue
        
        try:
            print(f"\n📊 Batch {batch_idx + 1}/{len(training_batches)}: {batch['batch_id']}")
            print(f"   Processing {len(batch_data)} locations...")
            
            start_time = time.time()
            
            features = extract_landsat_features_batch(batch_data, buffer_meters=BUFFER_SIZE)
            features.to_csv(output_file, index=False)
            
            elapsed = time.time() - start_time
            successful_batches += 1
            
            print(f"   ✅ Saved ({elapsed:.1f}s) - Progress: {successful_batches}/{len(training_batches)}")
            
            # Progress update every 10 batches
            if (batch_idx + 1) % 10 == 0:
                progress_pct = (successful_batches / len(training_batches)) * 100
                print(f"\n🎯 PROGRESS UPDATE: {progress_pct:.1f}% complete ({successful_batches}/{len(training_batches)} batches)")
        
        except Exception as e:
            print(f"   ❌ Error: {e}")
            failed_batches.append(batch['batch_id'])
            continue
    
    print(f"\n🎉 50m buffer training extraction COMPLETE!")
    print(f"✅ Successful: {successful_batches}/{len(training_batches)} batches")
    if failed_batches:
        print(f"❌ Failed batches: {failed_batches}")
    
else:
    print("⚠️ Run the training batch preparation cell first!")

### 🔴 Training Dataset: 150m Buffer Extraction
**Larger spatial context for all 9,319 training data points**

In [ ]:
# TRAINING 150m BUFFER - ALL BATCHES
# Process all training batches for 150m buffer
# Run after completing 50m buffer extraction

if 'training_batches' in globals():
    BUFFER_SIZE = 150
    
    print(f"🚀 Starting 150m buffer extraction for ALL training batches...")
    print(f"Total batches: {len(training_batches)} (≈{len(training_batches) * 3.5:.1f} hours)")
    
    successful_batches = 0
    failed_batches = []
    
    for batch_idx, batch in enumerate(training_batches):
        batch_data = batch['data']
        
        output_file = f"landsat_train_{BUFFER_SIZE}m_{batch['batch_id']}.csv"
        
        # Skip if already exists
        if os.path.exists(output_file):
            print(f"⏭️  Batch {batch_idx + 1}: {output_file} already exists, skipping...")
            successful_batches += 1
            continue
        
        try:
            print(f"\n📊 Batch {batch_idx + 1}/{len(training_batches)}: {batch['batch_id']}")
            print(f"   Processing {len(batch_data)} locations with {BUFFER_SIZE}m buffer...")
            
            start_time = time.time()
            
            features = extract_landsat_features_batch(batch_data, buffer_meters=BUFFER_SIZE)
            features.to_csv(output_file, index=False)
            
            elapsed = time.time() - start_time
            successful_batches += 1
            
            print(f"   ✅ Saved ({elapsed:.1f}s) - Progress: {successful_batches}/{len(training_batches)}")
            
            # Progress update every 2 batches
            if (batch_idx + 1) % 2 == 0:
                progress_pct = (successful_batches / len(training_batches)) * 100
                print(f"\n🎯 PROGRESS UPDATE: {progress_pct:.1f}% complete ({successful_batches}/{len(training_batches)} batches)")
        
        except Exception as e:
            print(f"   ❌ Error: {e}")
            failed_batches.append(batch['batch_id'])
            continue
    
    print(f"\n🎉 150m buffer training extraction COMPLETE!")
    print(f"✅ Successful: {successful_batches}/{len(training_batches)} batches")
    if failed_batches:
        print(f"❌ Failed batches: {failed_batches}")
    
else:
    print("⚠️ Run the training batch preparation cell first!")

### 🔴 Training Dataset: 200m Buffer Extraction  
**Maximum spatial context for landscape-scale influences**

In [ ]:
# TRAINING 200m BUFFER - ALL BATCHES
# Process all training batches for 200m buffer
# Run after completing 50m and 150m buffer extractions

if 'training_batches' in globals():
    BUFFER_SIZE = 200
    
    print(f"🚀 Starting 200m buffer extraction for ALL training batches...")
    print(f"Total batches: {len(training_batches)} (≈{len(training_batches) * 3.5:.1f} hours)")
    
    successful_batches = 0
    failed_batches = []
    
    for batch_idx, batch in enumerate(training_batches):
        batch_data = batch['data']
        
        output_file = f"landsat_train_{BUFFER_SIZE}m_{batch['batch_id']}.csv"
        
        # Skip if already exists
        if os.path.exists(output_file):
            print(f"⏭️  Batch {batch_idx + 1}: {output_file} already exists, skipping...")
            successful_batches += 1
            continue
        
        try:
            print(f"\n📊 Batch {batch_idx + 1}/{len(training_batches)}: {batch['batch_id']}")
            print(f"   Processing {len(batch_data)} locations with {BUFFER_SIZE}m buffer...")
            
            start_time = time.time()
            
            features = extract_landsat_features_batch(batch_data, buffer_meters=BUFFER_SIZE)
            features.to_csv(output_file, index=False)
            
            elapsed = time.time() - start_time
            successful_batches += 1
            
            print(f"   ✅ Saved ({elapsed:.1f}s) - Progress: {successful_batches}/{len(training_batches)}")
            
            # Progress update every 2 batches
            if (batch_idx + 1) % 2 == 0:
                progress_pct = (successful_batches / len(training_batches)) * 100
                print(f"\n🎯 PROGRESS UPDATE: {progress_pct:.1f}% complete ({successful_batches}/{len(training_batches)} batches)")
        
        except Exception as e:
            print(f"   ❌ Error: {e}")
            failed_batches.append(batch['batch_id'])
            continue
    
    print(f"\n🎉 200m buffer training extraction COMPLETE!")
    print(f"✅ Successful: {successful_batches}/{len(training_batches)} batches")
    if failed_batches:
        print(f"❌ Failed batches: {failed_batches}")
    
else:
    print("⚠️ Run the training batch preparation cell first!")

#### 📋 Final Training Assembly - Combine All Training Batches
**Run after all training extractions complete to create final training feature files**

In [ ]:
# COMBINE ALL TRAINING BATCHES - Create Final Training Feature Files
# Run this after all training extractions complete

import glob
from pathlib import Path

def combine_training_landsat_batches():
    """Combine all training batch CSV files into final feature datasets"""
    
    print("🔄 Combining all TRAINING Landsat extraction batches...\n")
    
    for buffer in [50, 150, 200]:
        pattern = f'landsat_train_{buffer}m_train_batch_*.csv'
        batch_files = sorted(glob.glob(pattern))
        
        print(f"📊 {buffer}m buffer: Found {len(batch_files)} training batch files")
        
        if not batch_files:
            print(f"   ⚠️ No training batch files found for {buffer}m buffer")
            continue
        
        # Load and combine all batches
        all_batches = []
        total_locations = 0
        
        for file in batch_files:
            try:
                batch = pd.read_csv(file)
                all_batches.append(batch)
                total_locations += len(batch)
                print(f"   ✅ {Path(file).name}: {len(batch)} locations")
            except Exception as e:
                print(f"   ❌ Error loading {Path(file).name}: {e}")
        
        if all_batches:
            # Combine all batches
            combined = pd.concat(all_batches, ignore_index=True)
            
            # Remove duplicates (if any)
            before = len(combined)
            combined = combined.drop_duplicates(subset=['Latitude', 'Longitude', 'Sample Date']).reset_index(drop=True)
            after = len(combined)
            
            if before > after:
                print(f"   🔧 Removed {before-after} duplicate rows")
            
            # Save combined dataset (matches benchmark expectations)
            output_file = f'landsat_features_training_{buffer}m.csv'
            combined.to_csv(output_file, index=False)
            
            print(f"   🎉 {output_file} saved: {len(combined)} locations × {len(combined.columns)} features")
            
            # Display sample
            print(f"   📋 Sample data:")
            print(combined.head(2).to_string(index=False))
        
        print()
    
    print("✅ Multi-buffer TRAINING extraction COMPLETE!")
    print("\n📁 Final training files created:")
    for buffer in [50, 150, 200]:
        filename = f'landsat_features_training_{buffer}m.csv'
        if Path(filename).exists():
            size = Path(filename).stat().st_size / 1024 / 1024  # MB
            print(f"   • {filename} ({size:.1f} MB)")
    
    print(f"\n🚀 Next: Use these files in Modeling_Optimization.ipynb for multi-buffer experiments!")

# Run the combination
combine_training_landsat_batches()

---

## 📋 Complete Extraction Workflow Summary

### ✅ COMPLETED: Validation Dataset (186 locations)
- **50m buffer**: landsat_features_50m_combined.csv  
- **150m buffer**: landsat_features_150m_combined.csv  
- **200m buffer**: landsat_features_200m_combined.csv

### 🔄 IN PROGRESS: Training Dataset (9,319 data points)

**Step 1**: Run preparation cell to create batches  
**Step 2**: Execute training extractions:
- `TRAINING 50m BUFFER - BATCH 1` → Test first batch (~3-4 hours)  
- `TRAINING 50m BUFFER - ALL REMAINING BATCHES` → Complete 50m (~35+ hours total)  
- `TRAINING 150m BUFFER - ALL BATCHES` → Complete 150m (~35+ hours)  
- `TRAINING 200m BUFFER - ALL BATCHES` → Complete 200m (~35+ hours)  

**Step 3**: Run final assembly to create:  
- `landsat_features_training_50m.csv`  
- `landsat_features_training_150m.csv`  
- `landsat_features_training_200m.csv`  

### ⚡ **KEY IMPROVEMENTS with 1000/batch:**
- **Reduced batches**: ~10 instead of ~186 batches per buffer  
- **More manageable**: Can complete in sessions vs. continuous monitoring  
- **Better progress tracking**: Every 2 batches instead of every 10  
- **Estimated time**: ~35 hours per buffer (down from ~93 hours)  

### 🎯 **Next Steps:**
1. Start with first 50m batch to test 1000-location performance  
2. Monitor extraction time per batch (should be 3-4 hours)  
3. Continue with remaining batches when ready  
4. Use extracted features in [Modeling_Optimization.ipynb](Modeling_Optimization.ipynb) for multi-buffer analysis